In [1]:
import requests
import pandas as pd
import os
from pathlib import Path
from dotenv import load_dotenv

# Raíz del proyecto: sube desde el directorio de trabajo hasta encontrar .git o pyproject.toml
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / ".git").exists() and not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
    
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
PROCESSED_PATH = PROCESSED_DATA / "cobre_clean.parquet"

In [2]:
load_dotenv(PROJECT_ROOT / ".env")

API_URL = os.getenv("COBRE_API_BC_URL", "").strip()
API_TOKEN = os.getenv("COBRE_API_BC_TOKEN", "").strip()

# Validacion de variables de entorno encontradas
if not API_URL:
    raise ValueError("Falta COBRE_API_BC_URL en el archivo .env")
if not API_TOKEN:
    raise ValueError("Falta COBRE_API_BC_TOKEN en el archivo .env")

PARAMS = {
    "token": API_TOKEN,
    "firstdate": "2024-01-01",
    "timeseries": "F019.PPB.PRE.100.D",
    "function": "GetSeries",
}

# Respeuesta y validaciones de respuesta desde API
response = requests.get(API_URL, params=PARAMS, timeout=30)
response.raise_for_status() 

data = response.json()

if data.get("Codigo") != 0:
    raise ValueError(f"Error API BCCH ({data.get('Codigo')}): {data.get('Descripcion')}")

obs = data.get("Series", {}).get("Obs")
if not obs:
    raise ValueError("La respuesta de la API no contiene datos en Series.Obs")

df = pd.DataFrame(obs)


df.head()


,indexDateString,value,statusCode
0,02-01-2024,3.8693,OK
1,03-01-2024,3.8392,OK
2,04-01-2024,3.8286,OK
3,05-01-2024,3.8186,OK
4,06-01-2024,NaN,ND


### Analisis del DataFrame obtenido con la API

In [3]:
df_a = df.copy()
df_a.head()

,indexDateString,value,statusCode
0,02-01-2024,3.8693,OK
1,03-01-2024,3.8392,OK
2,04-01-2024,3.8286,OK
3,05-01-2024,3.8186,OK
4,06-01-2024,NaN,ND


In [4]:
df_a.shape

(1057, 3)

In [5]:
df_a.info()

<class 'pandas.DataFrame'>
RangeIndex: 1057 entries, 0 to 1056
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   indexDateString  1057 non-null   str  
 1   value            1057 non-null   str  
 2   statusCode       1057 non-null   str  
dtypes: str(3)
memory usage: 42.4 KB


In [6]:
df_a.isna().sum()

indexDateString    0
value              0
statusCode         0
dtype: int64

In [7]:
df_a.duplicated().sum()

np.int64(0)

In [8]:
df_a.describe(include='all')

,indexDateString,value,statusCode
count,1057,1057,1057
unique,1057,699,2
top,02-01-2024,NaN,OK
freq,1,336,721


### Transformacion y limpieza

In [9]:
df_w = df_a.copy()

In [10]:
# Limpieza básica de tipos
df_w["fecha"] = pd.to_datetime(df_w["indexDateString"], format="%d-%m-%Y")
df_w["cobre_observado"] = pd.to_numeric(df_w["value"], errors="coerce")

In [11]:
df_w.columns

Index(['indexDateString', 'value', 'statusCode', 'fecha', 'cobre_observado'],
      dtype='str')

In [12]:
df_w = df_w.drop(columns=['indexDateString', 'value', 'statusCode'])

df_w.head()

,fecha,cobre_observado
0,2024-01-02,3.8693
1,2024-01-03,3.8392
2,2024-01-04,3.8286
3,2024-01-05,3.8186
4,2024-01-06,NaN


In [13]:
df_w.dtypes

fecha              datetime64[us]
cobre_observado           float64
dtype: object

In [14]:
df_w.shape

(1057, 2)

In [15]:
df_w['dia'] = df_w['fecha'].dt.day
df_w['mes'] = df_w['fecha'].dt.month
df_w['anio'] = df_w['fecha'].dt.year

TRADUCCION = {
    'Monday': 'lunes',
    'Tuesday': 'martes',
    'Wednesday': 'miercoles',
    'Thursday': 'jueves',
    'Friday': 'viernes',
    'Saturday': 'sabado',
    'Sunday': 'domingo',
}

df_w['dia_semana'] = df_w['fecha'].dt.day_name().map(TRADUCCION)

In [16]:
df_w.head()

,fecha,cobre_observado,dia,mes,anio,dia_semana
0,2024-01-02,3.8693,2,1,2024,martes
1,2024-01-03,3.8392,3,1,2024,miercoles
2,2024-01-04,3.8286,4,1,2024,jueves
3,2024-01-05,3.8186,5,1,2024,viernes
4,2024-01-06,NaN,6,1,2024,sabado


#### Corroborar el porque de los NaN en cobre observado segun las fechas para confirmar que se deben a dias fin de semana y festivos

In [17]:
# son 336 valores NaN
df_w['cobre_observado'].isna().sum()

np.int64(336)

In [18]:
df_corroborar_nan = df_w[df_w['dia_semana'].isin(['sabado', 'domingo'])].copy()
df_corroborar_nan.shape
# 302 registros corresponden a sabado y domingo

(302, 6)

In [19]:
df_corroborar_nan['cobre_observado'].isna().sum()
# de los 302 resgistros sabados y domingo 302 son sabado y domingo

np.int64(302)

In [20]:
# Todos los dias sabados y domingo tienen NaN como valor de dolar, se eliminan esas fila
df_corroborar_nan = df_w[~df_w['dia_semana'].isin(['sabado', 'domingo'])].reset_index(drop=True)

In [21]:
df_corroborar_nan[df_corroborar_nan['cobre_observado'].isna()]
# quedan 34 registros con NaN

,fecha,cobre_observado,dia,mes,anio,dia_semana
63,2024-03-29,NaN,29,3,2024,viernes
86,2024-05-01,NaN,1,5,2024,miercoles
100,2024-05-21,NaN,21,5,2024,martes
122,2024-06-20,NaN,20,6,2024,jueves
140,2024-07-16,NaN,16,7,2024,martes
162,2024-08-15,NaN,15,8,2024,jueves
186,2024-09-18,NaN,18,9,2024,miercoles
187,2024-09-19,NaN,19,9,2024,jueves
188,2024-09-20,NaN,20,9,2024,viernes
217,2024-10-31,NaN,31,10,2024,jueves


Son 34 registros en total (13 en 2024, 13 en 2025 y 8 en 2026), y el 100% corresponde a días inhábiles financieros en Chile: 32 son feriados legales nacionales caídos en día de semana  y 2 corresponden al Feriado Bancario (31 de diciembre), día en que ni los bancos ni el Banco Central operan, lo que explica exactamente los valores

In [22]:
df_corroborar_nan = df_corroborar_nan.dropna(subset=['cobre_observado'])

In [23]:
df_corroborar_nan['cobre_observado'].isna().sum()

np.int64(0)

Se corrobora informacion, NaN del dataset corresponden unicamente a sabados, domingo y festivos en chile.

### Limpieza Data set de trabajo

In [24]:
df_w = (
    df_w.dropna(subset=['cobre_observado'])
    .sort_values('fecha')
    .reset_index(drop=True)
)

df_w['cobre_observado'].isna().sum()


np.int64(0)

In [25]:
print('Resumen del Dataset Work luego de transformaciones y limpieza')
print('=============================================================')
print('DF_Analysis: ', df_a.shape, "-> DF_Work: ", df_w.shape)
print('=============================================================')
print('DF_Work tiene: ', df_w.isna().sum().sum(), ' Valores Nulos')
print('=============================================================')
print('DF_Work tiene: ', df_w.duplicated().sum(), ' Valores Duplicados')

Resumen del Dataset Work luego de transformaciones y limpieza
DF_Analysis:  (1057, 3) -> DF_Work:  (721, 6)
DF_Work tiene:  0  Valores Nulos
DF_Work tiene:  0  Valores Duplicados


In [26]:
df_w = df_w[['fecha', 'anio', 'mes', 'dia', 'dia_semana', 'cobre_observado'] ]
df_w.head()


,fecha,anio,mes,dia,dia_semana,cobre_observado
0,2024-01-02,2024,1,2,martes,3.8693
1,2024-01-03,2024,1,3,miercoles,3.8392
2,2024-01-04,2024,1,4,jueves,3.8286
3,2024-01-05,2024,1,5,viernes,3.8186
4,2024-01-08,2024,1,8,lunes,3.7956


In [27]:
# Guardar el dataset transformado en formato Parquet
PROCESSED_DATA.mkdir(parents=True, exist_ok=True)
temporary = PROCESSED_PATH.with_name(f"{PROCESSED_PATH.name}.part")
df_w.to_parquet(temporary, index=False, engine="pyarrow")
temporary.replace(PROCESSED_PATH)

print(f"Dataset procesado: {PROCESSED_PATH}")

Dataset procesado: /Users/diegovidelasilva/DAVS_DataScience/Proyectos/proyecto_etl_cobre_dolar_chile/data/processed/cobre_clean.parquet
